In [1]:
include("../RayTracing.jl")

Main.RayTracing

# Calculating intersection & normals with PBRT

### Case 1: hitting the closest point

In [2]:
center = RayTracing.Pnt3(1,1,1)
radius = 2.0

ray = RayTracing.Ray(
    RayTracing.Pnt3(10, 10, 10),
    RayTracing.Vec3(-1, -1, -1),
    0.0,
    999999999.0
)

sphere_transform = RayTracing.Translate(center)
sphere = RayTracing.Sphere(
    RayTracing.ShapeCore(
        sphere_transform,
        RayTracing.Inv(sphere_transform),
        false,
        false
    ),
    radius
)

check, t, si = RayTracing.intersect(sphere, ray)

@assert si.core.p ≈ RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)
@assert si.core.n ≈ RayTracing.Pnt3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)

### Case 2: hitting the 'corner' at x,y,0

In [3]:
center = RayTracing.Pnt3(1,1,1)
radius = 2.0

ray = RayTracing.Ray(
    RayTracing.Pnt3(10, 10, 10),
    RayTracing.Vec3(sqrt((radius^2)/2)+center.x, sqrt((radius^2)/2)+center.y, center.z) - RayTracing.Pnt3(10, 10, 10),
    0.0,
    999999999.0
)

sphere_transform = RayTracing.Translate(center)
sphere = RayTracing.Sphere(
    RayTracing.ShapeCore(
        sphere_transform,
        RayTracing.Inv(sphere_transform),
        false,
        false
    ),
    radius
)

check, t, si = RayTracing.intersect(sphere, ray)

@assert si.core.p ≈ RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)
@assert si.core.n ≈ RayTracing.Pnt3(0.7071067811865476, 0.7071067811865476, 0)

# OK! Now to confirm the normal math works!

In [4]:
function f(s::RayTracing.Sphere, p::RayTracing.Pnt3)::Float64
    center = s.core.object_to_world(RayTracing.Pnt3(0,0,0))
    return RayTracing.distance_squared(center, p) - s.radius^2
end

function normal_slow(s::RayTracing.Sphere, p::RayTracing.Pnt3)::RayTracing.Vec3
    h = 0.00001
    fx = (f(s, p + RayTracing.Pnt3(h,0,0)) - f(s, p + RayTracing.Pnt3(-h,0,0)))/(2.0 * h)
    fy = (f(s, p + RayTracing.Pnt3(0,h,0)) - f(s, p + RayTracing.Pnt3(0,-h,0)))/(2.0 * h)
    fz = (f(s, p + RayTracing.Pnt3(0,0,h)) - f(s, p + RayTracing.Pnt3(0,0,-h)))/(2.0 * h)
    return RayTracing.normalize(RayTracing.Vec3(fx, fy, fz))
end

function normal(s::RayTracing.Sphere, p::RayTracing.Pnt3)::RayTracing.Vec3
    e = .00001
    return RayTracing.normalize(
        RayTracing.Vec3(1, -1, -1) * f(s, p + RayTracing.Vec3(e, -e, -e)) +
        RayTracing.Vec3(-1, -1, 1) * f(s, p + RayTracing.Vec3(-e, -e, e)) +
        RayTracing.Vec3(-1, 1, -1) * f(s, p + RayTracing.Vec3(-e, e, -e)) +
        RayTracing.Vec3(1, 1, 1) * f(s, p + RayTracing.Vec3(e, e, e))
    )
end

normal_fast (generic function with 1 method)

In [5]:
@assert normal(sphere, RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)) ≈ RayTracing.Vec3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)
@assert normal(sphere, RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)) ≈ RayTracing.Vec3(0.7071067811865476, 0.7071067811865476, 0)

@assert normal_fast(sphere, RayTracing.Pnt3(2.1547005383792515, 2.1547005383792515, 2.1547005383792515)) ≈ RayTracing.Vec3(0.5773502691896258, 0.5773502691896258, 0.5773502691896258)
@assert normal_fast(sphere, RayTracing.Pnt3(2.414213562373095, 2.414213562373095, 1.0)) ≈ RayTracing.Vec3(0.7071067811865476, 0.7071067811865476, 0)

# Now, let's see if we can back into dpdu and dpdv and dndu dndv

In [13]:
n, v1, v2 = RayTracing.orthonormal_basis(RayTracing.Vec3(si.core.n))

([0.7071067811865476, 0.7071067811865476, 8.881784197001249e-16], [0.0, 1.2560739669470195e-15, -1.0], [-0.7071067811865476, 0.7071067811865476, 8.881784197001249e-16])